# P02.2 - LangUsta Custom Benchmark

A 100-question Turkish multiple-choice benchmark built from the held-out split excluded from LangUsta MCQ Letter LoRA training.

## 1. Environment

In [ ]:
!pip -q install --upgrade transformers datasets peft accelerate bitsandbytes sentencepiece protobuf


## 2. Configuration

In [ ]:
from pathlib import Path
import gc, hashlib, json, random, re, shutil, time
import torch
from datasets import Dataset, load_dataset
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

SEED = 3407
LETTERS = "ABCDE"
SOURCE_DATASET = "AhmetSemih/Deepseek-mcq-reasoning-dataset"
OUTPUT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/content")
OUTPUT_DIR = OUTPUT_ROOT / "langusta-custom-benchmark"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SYSTEM_PROMPT = (
    "Sen Türkçe çoktan seçmeli soruları yanıtlayan bir asistansın. "
    "Yanıt olarak yalnızca doğru seçeneğin A, B, C, D veya E harfini yaz. "
    "Açıklama, gerekçe veya başka bir metin ekleme."
)

MODEL_CONFIGS = [
    {"name": "LangUsta-MCQ-Letter-LoRA", "model_id": "Qwen/Qwen2.5-0.5B-Instruct", "adapter": True},
    {"name": "Qwen2.5-0.5B-Instruct", "model_id": "Qwen/Qwen2.5-0.5B-Instruct"},
    {"name": "Qwen2.5-1.5B-Instruct", "model_id": "Qwen/Qwen2.5-1.5B-Instruct"},
    {"name": "SmolLM2-1.7B-Instruct", "model_id": "HuggingFaceTB/SmolLM2-1.7B-Instruct"},
    {"name": "Gemma-3-1B-Instruct-4bit", "model_id": "unsloth/gemma-3-1b-it-unsloth-bnb-4bit", "prequantized": True},
]

assert torch.cuda.is_available(), "GPU bulunamadı. Kaggle Settings bölümünden GPU seçin."
print(f"Device: {torch.cuda.get_device_name(0)}")


## 3. Held-Out Benchmark Construction

In [ ]:
def clean_text(value):
    return "" if value is None else " ".join(str(value).replace("\x00", " ").split()).strip()

def normalize(value):
    value = clean_text(value).casefold().replace("’", "'").replace("`", "'")
    return re.sub(r"[^\wçğıöşü]+", " ", value, flags=re.UNICODE).strip()

def answer_letter(answer, options):
    raw = clean_text(answer).upper()
    if raw in LETTERS and LETTERS.index(raw) < len(options):
        return raw
    matches = [index for index, option in enumerate(options) if normalize(option) == normalize(answer)]
    return LETTERS[matches[0]] if len(matches) == 1 and matches[0] < 5 else None

def question_prompt(question, options):
    choices = "\n".join(f"{LETTERS[index]}: {option}" for index, option in enumerate(options))
    return f"Soru: {question}\n\nSeçenekler:\n{choices}\n\nYalnızca doğru seçeneğin harfini yaz."

source = load_dataset(SOURCE_DATASET, split="train")
records, seen = [], set()
for row in source:
    question = clean_text(row.get("question"))
    options = [clean_text(option) for option in (row.get("options") or [])]
    label = answer_letter(row.get("answer"), options)
    key = normalize(question)
    if not question or len(options) != 5 or any(not item for item in options) or not label or key in seen:
        continue
    seen.add(key)
    records.append({
        "id": hashlib.sha256(key.encode("utf-8")).hexdigest()[:16],
        "section": clean_text(row.get("section")),
        "topic": clean_text(row.get("topic")),
        "question": question,
        "options": options,
        "answer": label,
    })

records.sort(key=lambda item: item["id"])
rng = random.Random(SEED)
benchmark_records = []
for letter in LETTERS:
    group = [record for record in records if record["answer"] == letter]
    rng.shuffle(group)
    benchmark_records.extend(group[:max(1, round(len(group) * 0.10))])
rng.shuffle(benchmark_records)

assert len(records) == 997, f"Beklenen 997 geçerli kayıt yerine {len(records)} bulundu."
assert len(benchmark_records) == 100, f"Benchmark 100 soru olmalı, bulunan: {len(benchmark_records)}"
assert {item["answer"] for item in benchmark_records} == set(LETTERS)

benchmark_path = OUTPUT_DIR / "langusta_mcq_benchmark.jsonl"
benchmark_path.write_text(
    "\n".join(json.dumps(item, ensure_ascii=False) for item in benchmark_records) + "\n",
    encoding="utf-8",
)
print(f"Benchmark records: {len(benchmark_records)}")


## 4. Adapter Discovery

In [ ]:
def find_adapter():
    input_root = Path("/kaggle/input")
    configs = list(input_root.rglob("adapter_config.json")) if input_root.exists() else []
    for config in configs:
        if (config.parent / "adapter_model.safetensors").exists():
            return config.parent
    zips = list(input_root.rglob("langusta-mcq-letter-lora.zip")) if input_root.exists() else []
    if zips:
        destination = OUTPUT_ROOT / "uploaded-langusta-adapter"
        shutil.unpack_archive(str(zips[0]), destination)
        extracted = list(destination.rglob("adapter_config.json"))
        if extracted:
            return extracted[0].parent
    raise FileNotFoundError("LangUsta adaptörü bulunamadı. Çıkardığınız klasörü veya ZIP dosyasını Kaggle Input olarak ekleyin.")

ADAPTER_PATH = find_adapter()
print("Adapter loaded")


## 5. Six-Model Evaluation

In [ ]:
quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

def load_system(config):
    tokenizer_source = str(ADAPTER_PATH) if config.get("adapter") else config["model_id"]
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_source)
    model_kwargs = {"device_map": "auto", "low_cpu_mem_usage": True}
    if not config.get("prequantized"):
        model_kwargs["quantization_config"] = quantization
    model = AutoModelForCausalLM.from_pretrained(config["model_id"], **model_kwargs)
    if config.get("adapter"):
        model = PeftModel.from_pretrained(model, ADAPTER_PATH)
    model.eval()
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id
    return model, tokenizer

def render_prompt(tokenizer, record):
    user_content = question_prompt(record["question"], record["options"])
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        fallback = [{"role": "user", "content": SYSTEM_PROMPT + "\n\n" + user_content}]
        return tokenizer.apply_chat_template(fallback, tokenize=False, add_generation_prompt=True)

def extract_letter(text):
    match = re.match(r"^\s*([A-E])(?:\b|[\s:).\-]|$)", text.upper())
    return match.group(1) if match else None

all_predictions, summaries = [], []
for config in MODEL_CONFIGS:
    model, tokenizer = load_system(config)
    started = time.time()
    model_predictions = []

    for index, record in enumerate(benchmark_records, start=1):
        prompt = render_prompt(tokenizer, record)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.inference_mode():
            output = model.generate(
                **inputs,
                max_new_tokens=4,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        raw = tokenizer.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        predicted = extract_letter(raw)
        item = {
            "model": config["name"], "id": record["id"], "section": record["section"],
            "expected": record["answer"], "predicted": predicted, "raw_output": raw,
            "correct": predicted == record["answer"],
        }
        model_predictions.append(item)
        if index % 20 == 0:

    duration = time.time() - started
    correct = sum(item["correct"] for item in model_predictions)
    valid = sum(item["predicted"] is not None for item in model_predictions)
    summaries.append({
        "model": config["name"], "questions": 100, "correct": correct,
        "accuracy": round(correct, 2), "valid_format_rate": round(valid, 2),
        "runtime_seconds": round(duration, 2),
    })
    all_predictions.extend(model_predictions)
    print(f"{config['name']}: {correct}% accuracy, {valid}% valid format")

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()


## 6. Report and Export

In [ ]:
predictions_path = OUTPUT_DIR / "predictions.jsonl"
predictions_path.write_text(
    "\n".join(json.dumps(item, ensure_ascii=False) for item in all_predictions) + "\n",
    encoding="utf-8",
)
(OUTPUT_DIR / "summary.json").write_text(json.dumps(summaries, ensure_ascii=False, indent=2), encoding="utf-8")
dataset_card = """---
language: [tr]
license: apache-2.0
task_categories: [question-answering]
tags: [benchmark, multiple-choice, turkish, langusta]
pretty_name: LangUsta Custom Benchmark
---
# LangUsta Custom Benchmark

A 100-question Turkish multiple-choice benchmark created from the held-out split of `AhmetSemih/Deepseek-mcq-reasoning-dataset`. Records in this split were not used during LangUsta MCQ Letter LoRA training. Answers use `A`-`E` labels. Source license: Apache 2.0.
"""
(OUTPUT_DIR / "dataset-README.md").write_text(dataset_card, encoding="utf-8")

ranking = sorted(summaries, key=lambda item: item["accuracy"], reverse=True)
lines = [
    "# LangUsta Custom Benchmark Results", "",
    "The benchmark contains 100 Turkish five-choice questions held out before fine-tuning.",
    "All systems use greedy decoding and strict first-letter exact-match scoring.", "",
    "| Model | Correct | Accuracy | Runtime (s) |",
    "|---|---:|---:|---:|",
]
for item in ranking:
    lines.append(
        f"| {item['model']} | {item['correct']}/100 | {item['accuracy']:.0f}% | "
        f"{item['runtime_seconds']:.2f} |"
    )
(OUTPUT_DIR / "benchmark-results.md").write_text("\n".join(lines) + "\n", encoding="utf-8")

assert len(all_predictions) == len(MODEL_CONFIGS) * 100
archive = shutil.make_archive(str(OUTPUT_ROOT / "langusta-custom-benchmark"), "zip", root_dir=OUTPUT_DIR)
print("\n".join(lines))
print(f"\nReady: {archive}")
